# Taller 8 — Problema 5: Esfera conductora biseccionada
## Implementación numérica y visualización del potencial

**Universidad de Antioquia**  
**Curso:** Física Matemática I  

En este notebook se implementan los resultados analíticos del **Problema 5** del Taller 8. Una esfera conductora de radio $a$ se separa en dos hemisferios: el superior a potencial $V_0$ y el inferior a potencial cero. Se evalúa el potencial electrostático en el interior y en el exterior mediante la expansión en polinomios de Legendre y se construyen varias visualizaciones para explorar la solución.


## Importación de librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.special import eval_legendre
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f6f6f6",
    "axes.grid": True,
    "grid.alpha": 0.35,
    "lines.linewidth": 2.2,
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "legend.fontsize": 11,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

COLORS = plt.cm.tab10.colors
print("Librerías importadas")


## Parámetros y funciones del potencial

Las expresiones analíticas obtenidas en el taller son:

\[\phi_{\text{int}}(r,\theta)=\frac{V_0}{2}+\frac{V_0}{2}\sum_{n=0}^{\infty}\left[P_{2n}(0)-P_{2n+2}(0)\right]\left(\frac{r}{a}\right)^{2n+1}P_{2n+1}(\cos\theta)\]

\[\phi_{\text{ext}}(r,\theta)=\frac{V_0}{2}\left(\frac{a}{r}\right)+\frac{V_0}{2}\sum_{n=0}^{\infty}\left[P_{2n}(0)-P_{2n+2}(0)\right]\left(\frac{a}{r}\right)^{2n+2}P_{2n+1}(\cos\theta)\]

A continuación se implementan estas series con un número finito de términos.


In [ ]:
a = 1.0
V0 = 1.0
N_TERMS = 25


def odd_coeffs(n_terms, v0=V0):
    n = np.arange(n_terms)
    p_even = eval_legendre(2 * n, 0.0)
    p_even_next = eval_legendre(2 * n + 2, 0.0)
    coeffs = 0.5 * v0 * (p_even - p_even_next)
    l_vals = 2 * n + 1
    return l_vals, coeffs


def phi_int(r, theta, n_terms=N_TERMS, a=a, v0=V0):
    r, theta = np.broadcast_arrays(np.asarray(r, dtype=float), np.asarray(theta, dtype=float))
    x = np.cos(theta)
    phi = 0.5 * v0 * np.ones_like(r)
    l_vals, coeffs = odd_coeffs(n_terms, v0=v0)
    for l, c in zip(l_vals, coeffs):
        phi += c * (r / a) ** l * eval_legendre(l, x)
    return phi


def phi_ext(r, theta, n_terms=N_TERMS, a=a, v0=V0):
    r, theta = np.broadcast_arrays(np.asarray(r, dtype=float), np.asarray(theta, dtype=float))
    x = np.cos(theta)
    phi = 0.5 * v0 * (a / r)
    l_vals, coeffs = odd_coeffs(n_terms, v0=v0)
    for l, c in zip(l_vals, coeffs):
        phi += c * (a / r) ** (l + 1) * eval_legendre(l, x)
    return phi


def phi_total(r, theta, n_terms=N_TERMS, a=a, v0=V0):
    r, theta = np.broadcast_arrays(np.asarray(r, dtype=float), np.asarray(theta, dtype=float))
    phi = np.empty_like(r, dtype=float)
    inside = r <= a
    phi[inside] = phi_int(r[inside], theta[inside], n_terms=n_terms, a=a, v0=v0)
    phi[~inside] = phi_ext(r[~inside], theta[~inside], n_terms=n_terms, a=a, v0=v0)
    return phi


l_vals, coeffs = odd_coeffs(6)
for l, c in zip(l_vals, coeffs):
    print(f"l={l:2d}  C_l = {c:+.6f}")


## Potencial sobre la superficie ($r=a$)

Comparación entre la condición de frontera ideal y la serie truncada. Se observa el fenómeno de Gibbs cerca del ecuador.


In [ ]:
theta = np.linspace(0, np.pi, 600)
V_boundary = np.where(theta <= np.pi / 2, V0, 0.0)
V_series = phi_int(a, theta, n_terms=N_TERMS)

plt.figure(figsize=(8, 4.8))
plt.plot(theta, V_boundary, color=COLORS[0], linestyle="--", label="Condición ideal")
plt.plot(theta, V_series, color=COLORS[3], label=f"Serie con N={N_TERMS}")
plt.axvline(np.pi / 2, color="black", alpha=0.4)
plt.xlabel(r"Ángulo $	heta$")
plt.ylabel(r"$\phi(a,	heta)$")
plt.title("Potencial en la superficie de la esfera")
plt.legend()
plt.tight_layout()


## Mapa 2D del potencial (corte meridional)

El potencial se visualiza en el plano $(x,z)$ con simetría axial. Se resaltan líneas equipotenciales y la frontera $r=a$.


In [ ]:
r = np.linspace(0, 2.5 * a, 320)
theta = np.linspace(0, np.pi, 360)
R, Theta = np.meshgrid(r, theta)
Phi = phi_total(R, Theta, n_terms=N_TERMS)

X = R * np.sin(Theta)
Z = R * np.cos(Theta)

plt.figure(figsize=(7.5, 6.5))
levels = np.linspace(np.min(Phi), np.max(Phi), 40)
contour = plt.contourf(X, Z, Phi, levels=levels, cmap="coolwarm")
plt.contour(X, Z, Phi, levels=12, colors="k", alpha=0.35, linewidths=0.6)
plt.colorbar(contour, label=r"$\phi(r,	heta)$")

circle_theta = np.linspace(0, np.pi, 200)
plt.plot(a * np.sin(circle_theta), a * np.cos(circle_theta), color="black", linewidth=2)

plt.xlabel("x")
plt.ylabel("z")
plt.title("Potencial en corte meridional")
plt.axis("equal")
plt.tight_layout()


## Perfiles radiales para distintos ángulos

Se comparan las curvas $\phi(r)$ para varios valores de $	heta$.

In [ ]:
r_line = np.linspace(0.01 * a, 2.5 * a, 500)
angles = [0, np.pi / 6, np.pi / 3, np.pi / 2, 2 * np.pi / 3, 5 * np.pi / 6, np.pi]

plt.figure(figsize=(8, 5))
for idx, ang in enumerate(angles):
    phi_vals = np.where(r_line <= a,
                        phi_int(r_line, ang, n_terms=N_TERMS),
                        phi_ext(r_line, ang, n_terms=N_TERMS))
    plt.plot(r_line / a, phi_vals, color=COLORS[idx % len(COLORS)],
             label=fr"$	heta={ang/np.pi:.2f}\pi$")

plt.axvline(1.0, color="black", linestyle="--", alpha=0.5, label="r=a")
plt.xlabel(r"$r/a$")
plt.ylabel(r"$\phi(r,	heta)$")
plt.title("Perfiles radiales del potencial")
plt.legend(ncol=2)
plt.tight_layout()


## Convergencia de la serie en un punto

Se analiza la convergencia de la serie truncada en un punto representativo dentro de la esfera.

In [ ]:
r0 = 0.7 * a
theta0 = np.pi / 3
n_values = np.arange(1, 40)
phi_values = np.array([phi_int(r0, theta0, n_terms=n) for n in n_values])
phi_ref = phi_int(r0, theta0, n_terms=80)

plt.figure(figsize=(7.5, 4.5))
plt.plot(n_values, np.abs(phi_values - phi_ref), color=COLORS[2])
plt.yscale("log")
plt.xlabel("Número de términos N")
plt.ylabel(r"Error $|\phi_N-\phi_{\mathrm{ref}}|$")
plt.title("Convergencia de la serie (r=0.7a, θ=π/3)")
plt.tight_layout()


## Potencial sobre la superficie en 3D

Visualización de la distribución de potencial sobre la esfera usando la serie truncada.


In [ ]:
theta = np.linspace(0, np.pi, 120)
phi = np.linspace(0, 2 * np.pi, 240)
Theta, Phi = np.meshgrid(theta, phi)

X = a * np.sin(Theta) * np.cos(Phi)
Y = a * np.sin(Theta) * np.sin(Phi)
Z = a * np.cos(Theta)

V_surface = phi_int(a, Theta, n_terms=N_TERMS)

fig = plt.figure(figsize=(7.5, 6.0))
ax = fig.add_subplot(111, projection="3d")

norm = plt.Normalize(V_surface.min(), V_surface.max())
colors = cm.coolwarm(norm(V_surface))
ax.plot_surface(X, Y, Z, facecolors=colors, rstride=2, cstride=2, linewidth=0, antialiased=False)

mappable = cm.ScalarMappable(cmap="coolwarm", norm=norm)
mappable.set_array(V_surface)
fig.colorbar(mappable, shrink=0.65, pad=0.08, label=r"$\phi(a,	heta)$")

ax.set_title("Potencial sobre la superficie (3D)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=25, azim=35)
plt.tight_layout()
